# RegVelo 官方斑马鱼教学 / Official zebrafish walkthrough

这份 notebook 将模块的真实 CLI 与已保存证据连接起来，学习路线为 **audit → prepare → fit → velocity → fate → grn → perturb → visualize → report**。默认只读本机现有结果，不训练模型、不重算 velocity，也不修改正式运行状态。

**本 notebook 尚未执行（NOT_EXECUTED）。** 所有代码单元的 `execution_count` 为 `null`、`outputs` 为空。这里介绍的真实模型计算由独立 CLI/验证脚本完成，证据来自原有 `run_state.json`、QC、CSV 和图件元数据；它们不是 notebook 的运行输出。

截至编写时，正式 hard 配置保留全部 **697 cells**，预处理后 **1,007 genes / 82 effective regulators / 4,380 prior edges**；真实训练记录为 **624 epochs**。`audit` 至 `grn` 已有 PASS 记录。后续阶段以运行时读取的记录为准，不能由本段推定扰动或全部 Gallery 已完成。三个模式另有 **200 cells × 894 genes、2 epochs** 的 smoke 验证；这不能比较收敛、模式性能或生物学有效性。

English summary: A non-executed teaching notebook for the actual workflow and saved evidence. Existing model runs are separate from notebook execution; engineering PASS does not establish biological validity.

## 0. 先准备路径与环境 / Paths and environment

先读模块的 [README](../../README.md)、[新手指南](../../BEGINNER_GUIDE.md)、[数据要求](../../DATA_REQUIREMENTS.md)、[环境报告](../../ENVIRONMENT_REPORT.md) 和 [验证标准](../../MODEL_VALIDATION.md)。配置中的 `input`、`grn`、`output` 相对**配置文件所在目录**解析。

本机专用解释器位于与方法库同级的 `regvelo_runtime/.venv/Scripts/python.exe`。其他机器应按环境报告准备独立环境并修改下面的 `PYTHON`；已有环境无需重复安装。Notebook 的基础阅读单元只使用 Python 标准库；可选 H5AD/后验单元需要实际 RegVelo 环境作为 kernel。不要在原有 Seurat/Scanpy 环境里批量升级依赖。

如果从 GitHub 克隆后尚未放置大型数据或本机结果，读取单元会显示 `NOT_RUN` / missing file；公开代码和教程不包含原始 H5AD、模型、完整大矩阵与全部本机图件。

In [ ]:
from pathlib import Path
import csv
import hashlib
import itertools
import json
import math
import os
import shlex
import subprocess
import sys
from pprint import pprint

# 找不到模块时，把 None 改成真实模块目录；不从生成的工作目录名推断任何设置。
MODULE_ROOT = None
if MODULE_ROOT is None:
    here = Path.cwd().resolve()
    candidates = []
    for base in (here, *here.parents):
        candidates.extend([
            base,
            base / "03_cell_dynamics" / "RegVelo_GRN_Dynamics",
            base / "outputs" / "research-scripts-r" / "03_cell_dynamics" / "RegVelo_GRN_Dynamics",
        ])
    MODULE_ROOT = next((p for p in candidates if (p / "scripts" / "regvelo.py").is_file()
                        and (p / "config" / "official_zebrafish.json").is_file()), None)
    if MODULE_ROOT is None:
        raise FileNotFoundError("请将 MODULE_ROOT 设置为 RegVelo_GRN_Dynamics 的实际绝对路径。")
MODULE_ROOT = Path(MODULE_ROOT).resolve()

for name in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "NUMBA_NUM_THREADS"):
    os.environ.setdefault(name, "4")
sys.path.insert(0, str(MODULE_ROOT / "src"))
from regvelo_workflow.cli import STAGES, load_config

OFFICIAL_CONFIG = MODULE_ROOT / "config" / "official_zebrafish.json"
CONFIG = load_config(OFFICIAL_CONFIG)
OUTPUT = Path(CONFIG["output"])
SCRIPT = MODULE_ROOT / "scripts" / "regvelo.py"
runtime = MODULE_ROOT.parents[2] / "regvelo_runtime" / ".venv"
PYTHON = runtime / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
if not PYTHON.is_file():
    # 其他机器上仅在当前 kernel 已是 RegVelo 独立环境时使用它。
    PYTHON = Path(sys.executable)

pprint({"module": str(MODULE_ROOT), "config": str(OFFICIAL_CONFIG),
        "input": CONFIG["input"], "prior": CONFIG["grn"],
        "output": str(OUTPUT), "CLI_python": str(PYTHON),
        "notebook_kernel_python": sys.executable})


In [ ]:
def read_json(path):
    path = Path(path)
    if not path.is_file():
        return {"status": "NOT_RUN", "missing_file": str(path)}
    return json.loads(path.read_text(encoding="utf-8-sig"))


def read_csv(path, limit=None):
    path = Path(path)
    if not path.is_file():
        print("NOT_RUN: missing file", path)
        return []
    with path.open(encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        return list(reader if limit is None else itertools.islice(reader, limit))


def stage_record(stage):
    return read_json(OUTPUT / "run_state.json").get("stages", {}).get(stage, {"status": "NOT_RUN"})


# 默认空集：Run all 仅打印命令并读取现有结果。
# 执行某个阶段前先确认配置和独立输出；例如 {("audit", "official_zebrafish")}。
# --resume 只跳过同配置且完整产物存在的 PASS 阶段，不自动恢复 optimizer。
RUN_JOBS = set()


def run_stage(stage, config_file=OFFICIAL_CONFIG, tfs=(), resume=True):
    if stage not in (*STAGES, "all"):
        raise ValueError(f"Unknown CLI stage: {stage}")
    config_file = Path(config_file).resolve()
    command = [str(PYTHON), str(SCRIPT), stage, "--config", str(config_file)]
    if tfs:
        command += ["--tf", *tfs]
    if resume:
        command.append("--resume")
    print(subprocess.list2cmdline(command) if os.name == "nt" else shlex.join(command))
    if (stage, config_file.stem) in RUN_JOBS:
        subprocess.run(command, cwd=MODULE_ROOT, check=True)
    else:
        print("READ_ONLY: 命令未执行；需要执行时显式加入 RUN_JOBS。")


## 1. 读取真实运行状态 / Inspect execution evidence

每个阶段有独立的状态、耗时、解析配置 hash 与产物路径。`PASS` 只代表对应计算和工程校验完成；`REVIEW_REQUIRED`、`NOT_RUN` 与原始失败日志需要一起保留。文件存在不自动证明结果通过校验。

正式模型与 smoke 模型处于不同输出目录。改变 seed、预处理、终末定义、训练模式或 TF 面板时，应复制配置并指定新的独立 `output`。`--resume` 不能证明已中断训练的 optimizer 得到连续恢复；当前 `optimizer_resume` 明确为 `NOT_IMPLEMENTED`。

In [ ]:
state = read_json(OUTPUT / "run_state.json")
for stage in STAGES:
    record = state.get("stages", {}).get(stage, {"status": "NOT_RUN"})
    pprint({"stage": stage, "status": record.get("status"),
            "elapsed_seconds": record.get("elapsed_seconds"),
            "error": record.get("error"), "error_log": record.get("error_log")})
pprint(read_json(OUTPUT / "resolved_config.json"))


## 2. audit：检查数据与来源 / Audit inputs

官方 API 是 `rgv.datasets.zebrafish_nc()` / `rgv.datasets.zebrafish_grn()`。本例使用官方 **preprocessed** Smart-seq3 neural crest 对象，初始尺寸 697 × 8,012。其 `spliced` / `unspliced` 是非整数、已归一化层，不能再称为原始剪接计数；`matrix` 层也不能重命名为 raw spliced/unspliced。

实际对象的六个 somite stages 为 `3ss, 6-7ss, 10ss, 12-13ss, 17-18ss, 21-22ss`。顺序映射表示发育阶段 ordinal，不是 hpf，也不是经过时间。官方来源的说明、文件大小与 SHA 见 [来源清单](../../OFFICIAL_SOURCE_MANIFEST.csv) 和 [API 说明](../../references/API_NOTES.md)。原始研究的单细胞 GEO/SRA accession 尚未核实，不用旧 bulk RNA-seq accession 替代。

自有数据必须先核实真实剪接层的产生流程、物种与 gene ID；普通 expression/log1p/integrated 矩阵不能伪造 velocity。患者疾病组不能自动充当连续时间。

In [ ]:
run_stage("audit")
audit = read_json(OUTPUT / "INPUT_AUDIT.json")
pprint({key: audit.get(key) for key in ("n_cells", "n_genes", "layers", "obs_columns", "group_counts", "time_counts", "raw_counts_provenance", "readiness")})
pprint(read_csv(OUTPUT / "tables" / "input_cell_qc.csv", limit=5))
pprint(audit.get("sources", []))


In [ ]:
# 可选重新计算原文件 hash；只读，不写入审计或运行状态。
CHECK_SOURCE_HASHES = False
if CHECK_SOURCE_HASHES:
    for source in audit.get("sources", []):
        raw_path = Path(source["path"])
        if not raw_path.is_file():
            print("Missing source:", raw_path)
            continue
        digest = hashlib.sha256()
        with raw_path.open("rb") as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(chunk)
        pprint({"file": raw_path.name, "sha256": digest.hexdigest(),
                "matches_record": digest.hexdigest() == source["sha256"]})


## 3. prepare：名称对齐、moments 与 GRN 方向 / Prepare and align

模块按实际名称交集与速度基因规则筛选特征，保留所有正式细胞；`baseline_input.h5ad` 保存 RegVelo min-max moments 处理之前的基线。`input_preservation_qc.json` 可检查 spliced/unspliced 与来源是否精确一致，避免 scVelo 基线误用 RegVelo 缩放后的矩阵。

GRN 的生物方向是 **regulator → target**，但不同接口的矩阵方向不同：prior CSV 为 regulator × target，官方 `set_prior_grn` 接收其转置；AnnData skeleton 保存 regulator × target；模型 `fc1.weight` / Jacobian 为 target × regulator。不能仅因方阵就猜方向，必须查看 labelled-edge assertions。

来源 prior 是 4,508 × 4,508 方阵，但只有 **202 个非零 regulator rows**；方阵轴长度不是有效 TF 数。正式处理后是 82 个有效 regulators 与 4,380 个边。这里的 30 PCs 与官方教程的 50 PCs 不同，参数差异需要随结果保留。

自有命名边列表可配置 `grn_format="edge_list"`，默认读取 `source,target,weight`；自定义列名用 `grn_columns={"regulator":"source","target":"target","weight":"weight"}` 显式映射。`grn_orientation` 仍须声明。适配器保留输入权重符号、限定两端在输入基因轴内、拒绝重复 regulator–target 对；官方预处理生成二值 skeleton，不能把激活/抑制注释直接当作已验证模型方向。人类 CollecTRI 不能直接用于斑马鱼。


In [ ]:
run_stage("prepare")
pprint(read_json(OUTPUT / "prepare_qc.json"))
pprint(read_json(OUTPUT / "input_preservation_qc.json"))
pprint(read_csv(OUTPUT / "tables" / "aligned_gene_order.csv", limit=5))
pprint(read_csv(OUTPUT / "tables" / "grn_orientation_assertions.csv", limit=5))
pprint(read_csv(OUTPUT / "tables" / "feature_retention.csv", limit=5))


## 4. fit：官方 REGVELOVI 模型 / Fit the official model

正式配置是 hard、seed=0、max_epochs=1500、early stopping=true、batch_size=256。保存的实际训练记录为 624 epochs；batch size 与官方默认 full batch 不同，用于降低内存峰值。`history_finite`、`weights_finite`、`save_load_equal` 检查计算完整性，不等同于已经证明收敛或跨 seed 稳定。

`training_history.csv` 合并了 epoch 与稀疏记录的 step 指标，合并表中的缺失项不必然是训练数值 NaN。应读取各 `training_<metric>.csv` 的实际已记录值并结合 `fit_revalidation.json`；初始错误 QC 另存为 `fit_qc_initial.json`，不删掉原始记录。

In [ ]:
run_stage("fit")
fit_qc = read_json(OUTPUT / "fit_qc.json")
pprint(fit_qc)
pprint(read_json(OUTPUT / "fit_revalidation.json"))
pprint(read_csv(OUTPUT / "tables" / "training_train_loss_epoch.csv", limit=5))
pprint(read_csv(OUTPUT / "tables" / "training_validation_loss.csv", limit=5))


In [ ]:
# 只检查真实记录的指标值；空表和缺失值分别保留，避免把 join 缺失当作模型 NaN。
metric_checks = []
for path in sorted((OUTPUT / "tables").glob("training_*.csv")):
    if path.name == "training_history.csv":
        continue
    rows = read_csv(path)
    values = [float(value) for row in rows for key, value in row.items()
              if key and value not in ("", None)]
    metric_checks.append({"metric_file": path.name, "recorded_values": len(values),
                          "all_recorded_values_finite": all(math.isfinite(v) for v in values) if values else None})
pprint(metric_checks)


## 5. 三种模式 smoke / Hard, soft and soft_regularized

| mode | 官方约束参数 | 本次验证范围 |
|---|---|---|
| hard | `soft_constraint=False`, `lam2=0` | 200 cells × 894 genes，2 epochs |
| soft | `soft_constraint=True`, `lam=1`, `lam2=0` | 同上 |
| soft_regularized | `soft_constraint=True`, `lam=1`, `lam2=0.1` | 同上 |

三种模式的训练、保存/重载与官方 `get_velocity(n_samples=2)` 有独立 PASS 记录。它们使用同一细胞与特征，原始 hard prepared 文件 hash 没有改变；soft prepared 文件因注册 `_scvi_manager_uuid` 改变而字节 hash 不同，数值矩阵、全部层、名称、GRN 与非 scvi 元数据逐项精确一致。不能仅由文件 hash 差异推定表达数据改变。

初次验证错误是将训练支持的 `devices=1` 误传给 `REGVELOVI.load`；正确 load 不接受该参数。纠正后复用了已保存模型，没有再训练，初始失败记录保留。smoke 不检验正式 soft 模式性能、CellRank fate、收敛或跨 seed 鲁棒性。

如果要重跑，应复制对应配置并设置新输出，先核对相对路径，再按 `audit → prepare → fit` 执行。下面展示现有配置的有效命令，默认不会执行；新 soft smoke 的训练原先由独立验证脚本调用，不能因为没有 CLI `run_state.json` 就判定没有训练，训练证据在 `fit_qc.json` / `mode_smoke_qc.json`。

In [ ]:
smoke_summary = read_json(MODULE_ROOT / "tests" / "validation_logs" / "mode_smoke_summary.json")
pprint({key: smoke_summary.get(key) for key in ("status", "evaluation_scope", "source_unchanged", "retrained_during_revalidation", "unvalidated_scientific_questions")})
pprint(read_csv(MODULE_ROOT / "examples" / "official_zebrafish" / "smoke_mode_comparison.csv"))
for filename in ("smoke.json", "smoke_soft.json", "smoke_soft_regularized.json"):
    config_file = MODULE_ROOT / "config" / filename
    smoke_config = load_config(config_file)
    smoke_output = Path(smoke_config["output"])
    print("\n", filename, smoke_output)
    pprint(read_json(smoke_output / "mode_smoke_qc.json"))
    for stage in ("audit", "prepare", "fit"):
        run_stage(stage, config_file=config_file)


In [ ]:
# 可选：使用 notebook 的 RegVelo kernel，从 saved model 重新抽样检验接口。
# 默认关闭。已有 smoke_velocity.npy / mode_smoke_qc.json 足够用于教学阅读。
RUN_POSTERIOR_SMOKE = False
if RUN_POSTERIOR_SMOKE:
    import anndata as ad
    import numpy as np
    import regvelo as rgv
    from regvelo_workflow.dynamics import _seed

    for filename in ("smoke.json", "smoke_soft.json", "smoke_soft_regularized.json"):
        smoke_config = load_config(MODULE_ROOT / "config" / filename)
        smoke_output = Path(smoke_config["output"])
        data = ad.read_h5ad(smoke_output / "prepared.h5ad")
        _seed(smoke_config)
        model = rgv.REGVELOVI.load(str(smoke_output / "model"), data,
                                  accelerator=smoke_config["train"]["accelerator"])
        velocity = model.get_velocity(n_samples=2, return_numpy=True, batch_size=128)
        pprint({"mode": smoke_config["mode"], "shape": list(velocity.shape),
                "expected_shape": [data.n_obs, data.n_vars],
                "finite": bool(np.isfinite(velocity).all())})
        del model, data, velocity


## 6. velocity：后验速度、latent time 与基线 / Velocity and baseline

正式 `velocity.h5ad` 存 RegVelo 结果，`scvelo_stochastic.h5ad` 是从未经 RegVelo moments 缩放的 baseline input 计算的 scVelo stochastic 基线。后验抽样数是 30；该单 seed 的 posterior sampling 与“多个训练 seed 的稳定性”是不同问题。

QC 中保存了 velocity 是否有限、latent-time 范围，以及对**阶段 ordinal**的 Spearman 相关。相关性仅检查与已知阶段的一致性，不把 latent time 当真实经过时间，也不把 UMAP 箭头当实测谱系。`cell_dynamics.csv` 的 `latent_time` 与 `latent_time_posterior_mean` 是不同导出项，读取前先看列名。

In [ ]:
run_stage("velocity")
pprint(read_json(OUTPUT / "velocity_qc.json"))
pprint(read_csv(OUTPUT / "tables" / "cell_dynamics.csv", limit=5))
pprint(read_csv(OUTPUT / "tables" / "kinetic_rates.csv", limit=5))


In [ ]:
# 可选大矩阵检查；默认不载入 H5AD，不修改文件。
READ_LARGE_ARRAYS = False
if READ_LARGE_ARRAYS:
    import anndata as ad
    import numpy as np
    velocity_data = ad.read_h5ad(OUTPUT / "velocity.h5ad")
    pprint({"shape": list(velocity_data.shape), "layers": list(velocity_data.layers),
            "obsm": list(velocity_data.obsm),
            "velocity_finite": bool(np.isfinite(velocity_data.layers["velocity"]).all())})
    uncertainty_keys = [key for key in velocity_data.layers if "std" in key or "uncert" in key]
    for key in uncertainty_keys:
        values = np.asarray(velocity_data.layers[key])
        pprint({"layer": key, "shape": list(values.shape), "finite": bool(np.isfinite(values).all())})
    if not uncertainty_keys:
        print("No saved posterior-uncertainty layer; do not infer that it was computed.")
    del velocity_data


## 7. fate：CellRank / GPCCA 与概率守恒

正式配置使用 7 macrostates、每个 terminal state 30 cells、纯 VelocityKernel（`kernel_mix=1.0`）和 `solver=direct`。终末状态为 `mNC_head_mesenchymal, mNC_arch2, mNC_hox34, Pigment`；需要结合 neural crest 发育注释解释，不能直接套到患者横断面疾病组。

先看 `baseline_probability_validation.json`，再读取 CSV 逐行检查 finite、范围和 row sum=1。失败时导出的 `baseline_FAILED_raw_fate_probabilities.csv` 是诊断证据，不能替代通过校验的 baseline。模块保留失败诊断并通过直接求解获得当前有效结果，没有通过截断或任意归一化掩盖错误。

`commitment_score = 1 − H(p)/log2(number_of_lineages)` 表示命运概率集中的程度，不证明不可逆命运承诺。下面从已保存概率读取并计算这个**解释用分数**，不写入模型文件。

In [ ]:
run_stage("fate")
pprint(read_json(OUTPUT / "fate_qc.json"))
pprint(read_json(OUTPUT / "tables" / "baseline_probability_validation.json"))
terminal_cells = read_json(OUTPUT / "terminal_cell_sets.json")
if "missing_file" not in terminal_cells:
    pprint({name: len(cells) for name, cells in terminal_cells.items()})
pprint(read_csv(OUTPUT / "tables" / "cellrank_states.csv", limit=5))


In [ ]:
fate_rows = read_csv(OUTPUT / "tables" / "baseline_fate_probabilities.csv")
if fate_rows:
    lineage_names = [key for key in fate_rows[0] if key != "cell"]
    probabilities = [[float(row[name]) for name in lineage_names] for row in fate_rows]
    values = list(itertools.chain.from_iterable(probabilities))
    row_errors = [abs(sum(row) - 1) for row in probabilities]
    pprint({"n_cells": len(probabilities), "lineages": lineage_names,
            "finite": all(math.isfinite(v) for v in values),
            "range": [min(values), max(values)], "max_row_sum_error": max(row_errors),
            "terminal_column_order_matches": lineage_names == list(terminal_cells)})
    assert all(math.isfinite(v) and -1e-8 <= v <= 1 + 1e-8 for v in values)
    assert max(row_errors) < 1e-5
    commitment = [1 + sum(p * math.log2(p) for p in row if p > 0) / math.log2(len(lineage_names))
                  for row in probabilities]
    pprint({"commitment_range": [min(commitment), max(commitment)],
            "interpretation": "Probability concentration; not proof of irreversible commitment"})


## 8. grn：TF–target、regulon 与 prior 对照 / Inspect the GRN

`tf_target_edges.csv` 是按 cutoff=0.001 导出的模型 `fc1` 权重与 Jacobian。`prior_supported=True` 表示该连接有先验支持，`experimental_support=NOT_CHECKED` 保留尚未核实实验支持的事实。真实 hard 结果有 4,379 个超过 cutoff 的模型边、0 个新增非先验边；这是 hard 约束下的计算结果，不能据此宣布发现新调控。

模型权重和表达 Jacobian 是不同量。官方 API 对每个 group 的非零 Jacobian 分别做 mean-absolute normalization；`state_tf_jacobians.csv` 可以看组内相对模式，不能直接比较不同组的绝对调控强度。

In [ ]:
run_stage("grn")
pprint(read_json(OUTPUT / "GRN_QC.json"))
pprint(read_csv(OUTPUT / "tables" / "regulon_ranking.csv", limit=10))
edges = read_csv(OUTPUT / "tables" / "tf_target_edges.csv")
for tf in CONFIG["perturb"]["tfs"]:
    selected = [row for row in edges if row["regulator"] == tf]
    pprint({"TF": tf, "effective_edges_at_saved_cutoff": len(selected), "top_rows": selected[:5]})
pprint(read_csv(OUTPUT / "tables" / "state_tf_jacobians.csv", limit=3))


## 9. perturb：官方 regulon-level in silico KO

官方候选 TF 是 `elf1` / `nr2f5`，名称大小写敏感。模块调用官方 `in_silico_block_simulation(..., effects=0)` 阻断该 TF 的下游模型权重；参数名是 **effects**，不是 effect。它不是把 RNA counts 置零，更不是 CRISPR 实验。

paired baseline、NULL no-op、zero-effective-edge control 与 KO 使用相同 cells/features 和**冻结的 baseline terminal cell IDs**。在同一 load + posterior sampling 路径之前设 seed，配对后验抽样；KO 分别重算 velocity graph / UMAP velocity embedding 和 CellRank kernel，不能复制 baseline 的投影。

先读取阶段状态和 `perturb_qc.json`。只有对应阶段通过、NULL 和 negative control 合格、概率列顺序与终末定义一致时，才能解释保存的 KO 数值。`L2 velocity difference` 与 `1−cosine` 是模型速度分数；`KO−baseline fate probability` 是条件概率差，不是 expression log2FC。

官方 depletion likelihood 是 ROC-AUC / normalized U 型统计量：baseline=1、KO=0，>0.5 表示预测 depletion；它不是实验效应量。官方 ranksums p 值与每个 TF 内各 terminal states 的 BH 调整不能包装为真实实验显著性。当前 multi-seed / cutoff sensitivity 是否完成，以 QC 的 `scientific_qc` 字段为准。

In [ ]:
# 使用配置中 elf1 与 nr2f5；--tf 会覆盖面板，改变面板时使用独立输出。
run_stage("perturb")
pprint(stage_record("perturb"))
perturb_qc = read_json(OUTPUT / "perturb_qc.json")
pprint(perturb_qc)
if stage_record("perturb").get("status") == "PASS" and perturb_qc.get("status") == "PASS":
    pprint(read_csv(OUTPUT / "tables" / "perturbation_summary.csv"))
    pprint(read_csv(OUTPUT / "tables" / "perturbation_depletion.csv", limit=12))
else:
    print("Perturbation is not validated PASS; retain error/NOT_RUN and do not label KO results as complete.")


In [ ]:
# 以下只在有效 perturb PASS 后检查文件，不进行新模拟。
if stage_record("perturb").get("status") == "PASS" and perturb_qc.get("status") == "PASS":
    for tf in CONFIG["perturb"]["tfs"]:
        difference_rows = read_csv(OUTPUT / "tables" / f"{tf}_fate_difference.csv")
        if difference_rows:
            names = [key for key in difference_rows[0] if key != "cell"]
            assert names == list(terminal_cells), "KO 与 baseline lineage column order 不一致。"
            differences = [[float(row[name]) for name in names] for row in difference_rows]
            pprint({"TF": tf, "fate_difference_finite": all(math.isfinite(v) for row in differences for v in row),
                    "max_delta_row_sum_abs": max(abs(sum(row)) for row in differences)})
        pprint(read_csv(OUTPUT / "tables" / f"{tf}_cell_effect.csv", limit=5))
        blocked = read_csv(OUTPUT / "tables" / f"{tf}_blocked_edges.csv")
        pprint({"TF": tf, "blocked_edges": len(blocked), "preview": blocked[:5]})
    pprint({key: perturb_qc.get(key) for key in ("no_op_velocity_max_abs_difference", "no_op_fate_max_abs_difference", "no_effective_edges_control", "terminal_cells_frozen", "posterior_draws_paired_by_seed", "scientific_qc")})


## 10. visualize / report：真实 Gallery 与当前报告

`visualize` 从对应阶段已经保存的数值出图。每个图 ID 的 JSON、source CSV、生成命令、证据类型与限制要一起查看；图件 `PASS` 表示完成渲染，不证明生物学有效。缺数据的图保持 `NOT_RUN`，不调用官方教程图片填空。PNG 用于预览，SVG/PDF 用于编辑；公开 [Gallery](../../gallery/README.md) 可能只包含压缩代表图，本机输出保存高质量版本。

下面默认读取现有 `FIGURE_CATALOG.csv` 并只展示元数据为 PASS 且确有 PNG 的图。每次读取都是当前结果，不把 notebook 中的旧说明当最终状态。`report` 汇集已保存阶段状态为 `ANALYSIS_REPORT.md`。

In [ ]:
run_stage("visualize")
run_stage("report")
catalog = read_csv(OUTPUT / "FIGURE_CATALOG.csv")
pprint([{key: row.get(key) for key in ("id", "english_name", "chinese_name", "status", "evidence_label", "scientific_qc", "reason")} for row in catalog])
pprint(read_json(OUTPUT / "figures" / "RENDER_QA.json"))
report_path = OUTPUT / "ANALYSIS_REPORT.md"
if report_path.is_file():
    print(report_path.read_text(encoding="utf-8")[:12000])
else:
    print("NOT_RUN:", report_path)


In [ ]:
# 显示现有真实图件；不嵌入或伪造 notebook 执行输出，不生成新图。
from IPython.display import Image, Markdown, display

for figure_id in ("A01", "B01", "C01", "D01", "E03"):
    metadata = read_json(OUTPUT / "figures" / f"{figure_id}.json")
    image_path = OUTPUT / "figures" / f"{figure_id}.png"
    if metadata.get("status") == "PASS" and image_path.is_file():
        display(Markdown(f"**{figure_id}: {metadata.get('chinese_name', '')} / {metadata.get('english_name', '')}**"))
        display(Image(filename=str(image_path), width=850))
        pprint({key: metadata.get(key) for key in ("evidence_label", "scientific_question", "limitations", "scientific_qc", "method", "input_file_paths")})
    else:
        pprint({"figure": figure_id, "status": metadata.get("status", "NOT_RUN"),
                "reason": metadata.get("reason", metadata.get("missing_file"))})


## 11. 完整运行、恢复与自有数据 / Run, resume and transfer

已保存的真实结果足够完成本 notebook 的阅读。确需重跑全部阶段时，在独立配置/输出下运行：

```powershell
python scripts/regvelo.py all --config config/official_zebrafish.json --resume
```

`python` 应替换为环境报告中的解释器。`all` 的实际顺序来自 CLI `STAGES`，遇失败停止并保留日志。要查看具体错误，读 `logs/<stage>_error.log` 和 `logs/workflow.log`；不要删除 `run_state.json` 伪装成功。修复后只重跑依赖变化的必要阶段。

自有数据先复制配置，检查 `input/grn/output/group_key/time_key/time_order/species/gene ID/QC provenance` 的实际含义；配置未知字段不会自动产生科学验证。缺原始剪接层时先做审计与计数可行性评估，不能用普通表达矩阵凑成 velocity。横断面种植体周围炎数据需要独立的同谱系动态与终末证据；JUND 不预设为疾病驱动因子。外部 DoseDirKO 分数与这里的 velocity / fate shift 不同尺度，不能直接合并比较。

完成阅读后的自检：来源与层尺度可解释；GRN 名称及方向通过；正式与 smoke 分开；训练保存/重载和有限值通过；fate 概率守恒；KO terminal IDs/column order/paired posterior/control 通过；未完成的 seed/cutoff/生物学方向检查仍保留；每张图有真实数值来源与解释边界。

真实 CollecTRI adapter 检查已保存 40,877 个对齐边、JUND 105 个 targets（见 `tests/validation_logs/collectri_adapter.json`）；该检查只验证先验适配，不表示完成 RegVelo 动力学或疾病因果验证。


In [ ]:
# 最终只读核对；本单元不把 notebook 标记为已经执行或更新正式 QC。
checks = {
    "notebook_intended_mode": "READ_EXISTING_RESULTS",
    "official_output": str(OUTPUT),
    "configured_stage_order": list(STAGES),
    "mode_smoke_status": read_json(MODULE_ROOT / "tests" / "validation_logs" / "mode_smoke_summary.json").get("status"),
    "formal_stages": {stage: stage_record(stage).get("status", "NOT_RUN") for stage in STAGES},
    "fit_scientific_qc": read_json(OUTPUT / "fit_qc.json").get("scientific_qc"),
    "velocity_scientific_qc": read_json(OUTPUT / "velocity_qc.json").get("scientific_qc"),
    "perturb_scientific_qc": read_json(OUTPUT / "perturb_qc.json").get("scientific_qc"),
}
pprint(checks)
